In [39]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from shapely.ops import unary_union
import geopandas as gpd

In [40]:
cmaq_file = "LOCUSIdling_HDVSpatial/all_202307.nc"

grid_d02_file = "latlon_ChicagoLADCO_d03.nc"
grid_d03_file = "latlon_ChicagoLADCO_d03.nc"

epa_dir = Path("./")

start_dt = pd.Timestamp("2023-07-01 00:00")
end_dt   = pd.Timestamp("2023-07-31 23:00")

epa_code = ['42401','42602','44201','42101','88101']
var      = ['SO2','NO2','O3','CO','PM25_TOT']

# CMAP county shapefile
cmap_cty = gpd.read_file('C:/Users/x12la/Desktop/Scripts/CMAP_cty.shp')
cmap_cty = cmap_cty.to_crs('EPSG:4326')

In [41]:
ds = xr.open_dataset(cmaq_file)
print(ds)

grid_d02 = xr.open_dataset(grid_d02_file)
grid_d03 = xr.open_dataset(grid_d03_file)

lon_d02 = grid_d02["lon"].values
lat_d02 = grid_d02["lat"].values

lon_d03 = grid_d03["lon"].values
lat_d03 = grid_d03["lat"].values

llat, ulat = lat_d02.min(), lat_d02.max()
llon, ulon = lon_d02.min(), lon_d02.max()

<xarray.Dataset>
Dimensions:   (TSTEP: 744, LAY: 1, ROW: 288, COL: 315)
Dimensions without coordinates: TSTEP, LAY, ROW, COL
Data variables:
    SO2       (TSTEP, LAY, ROW, COL) float32 ...
    NO2       (TSTEP, LAY, ROW, COL) float32 ...
    NO        (TSTEP, LAY, ROW, COL) float32 ...
    O3        (TSTEP, LAY, ROW, COL) float32 ...
    CO        (TSTEP, LAY, ROW, COL) float32 ...
    PM25_TOT  (TSTEP, LAY, ROW, COL) float32 ...
    PM25_EC   (TSTEP, LAY, ROW, COL) float32 ...
    PM25_OC   (TSTEP, LAY, ROW, COL) float32 ...
    PM10      (TSTEP, LAY, ROW, COL) float32 ...
Attributes: (12/35)
    IOAPI_VERSION:             ioapi-3.2: $Id: init3.F90 247 2023-03-22 15:59...
    EXEC_ID:                   ????????????????                              ...
    FTYPE:                     1
    CDATE:                     2026075
    CTIME:                     132710
    WDATE:                     2026075
    ...                        ...
    UPNAM:                     COMBINE         
    

In [42]:
cmap_union = cmap_cty.unary_union

def label_in_d03(df, lon_col="Longitude", lat_col="Latitude"):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326"
    )
    return gdf.geometry.within(cmap_union).to_numpy()

In [43]:
def find_index(stn_lon, stn_lat, wrf_lon, wrf_lat):
    xx, yy = [], []
    for i in range(len(stn_lat)):
        abslat = np.abs(wrf_lat - stn_lat[i])
        abslon = np.abs(wrf_lon - stn_lon[i])
        c = np.maximum(abslon, abslat)
        args = np.where(c == np.min(c))
        xx.append(args[-2][0])
        yy.append(args[-1][0])
    return xx, yy

def convert_epa_units(df, species):
    """
    Convert EPA AQS observation units to match CMAQ units.
    """

    if "Units of Measure" not in df.columns:
        return df
    # Only convert gases where EPA might report ppm
    if species in ["O3", "CO", "NO2", "SO2"]:
        mask = df["Units of Measure"].str.contains("Parts per million", na=False)
        # Convert ppm → ppb
        df.loc[mask, "Sample Measurement"] *= 1000
    return df

def crop_epa(file):
    df = pd.read_csv(file)

    # Handle underscore headers in some AQS downloads per Stacy's original code
    if "Date_GMT" in df.columns:
        df.columns = [c.replace("_", " ") for c in df.columns]

    # coarse bbox crop (d02 bbox)
    df = df[(df["Latitude"]  >= llat) & (df["Latitude"]  <= ulat) &
            (df["Longitude"] >= llon) & (df["Longitude"] <= ulon)]

    # Build datetime and time-crop
    df["Datetime GMT"] = pd.to_datetime(df["Date GMT"] + " " + df["Time GMT"])
    df = df[(df["Datetime GMT"] >= start_dt) & (df["Datetime GMT"] <= end_dt)]
    return df

# adopted from Stacy's original code
def merge_species(species, epa_code):
    epa_file = epa_dir / f"hourly_{epa_code}_2023.csv"
    df = crop_epa(epa_file)
    df = convert_epa_units(df, species)

    # Unique station locations
    stations = df[["Latitude", "Longitude"]].drop_duplicates()

    dataframes = []
    t_index = pd.date_range(start_dt, end_dt, freq="1H")

    for _, row in stations.iterrows():
        stn_lat = row["Latitude"]
        stn_lon = row["Longitude"]

        tmp = df[(df.Latitude == stn_lat) & (df.Longitude == stn_lon)].copy()
        tmp.index = tmp["Datetime GMT"]

        # Resample numeric vs metadata exactly like your original script logic
        num_cols = tmp.select_dtypes(include="number").columns
        non_num_cols = tmp.select_dtypes(exclude="number").columns

        tmp_num = tmp[num_cols].resample("1H").mean().reindex(t_index)
        tmp_non = tmp[non_num_cols].resample("1H").first()

        tmp = pd.concat([tmp_num, tmp_non], axis=1)

        # Find nearest d02 grid cell for CMAQ extraction
        x, y = find_index([stn_lon], [stn_lat], lon_d02, lat_d02)
        x, y = x[0], y[0]

        # Basic bounds check (avoids index errors)
        if not (0 <= x < lon_d02.shape[0] and 0 <= y < lon_d02.shape[1]):
            continue

        # Pull CMAQ surface layer
        model = ds[species][:, 0, x, y].values
        tmp["CMAQ"] = model[:len(tmp)]

        # Observations
        tmp["OBS"] = tmp["Sample Measurement"]

        # Metadata for later grouping/filtering
        tmp["Species"] = species
        tmp["x"] = x
        tmp["y"] = y
        tmp["Latitude"] = stn_lat
        tmp["Longitude"] = stn_lon

        dataframes.append(tmp)
    return pd.concat(dataframes, ignore_index=False)

def calc_metrics(df):
    df = df.dropna(subset=["OBS", "CMAQ"])
    if len(df) == 0:
        return pd.Series({
            "N": 0,
            "Mean_OBS": np.nan,
            "Mean_CMAQ": np.nan,
            "MB": np.nan,
            "RMSE": np.nan,
            "R": np.nan,
            "NMB_%": np.nan,
            "NME_%": np.nan,
        })

    obs = df["OBS"].astype(float)
    mod = df["CMAQ"].astype(float)

    diff = mod - obs
    sum_obs = obs.sum()

    # Normalized metrics (percent), guard against divide-by-zero
    if sum_obs == 0:
        nmb = np.nan
        nme = np.nan
    else:
        nmb = 100.0 * diff.sum() / sum_obs
        nme = 100.0 * np.abs(diff).sum() / sum_obs

    return pd.Series({
        "N": len(df),
        "Mean_OBS": obs.mean(),
        "Mean_CMAQ": mod.mean(),
        "MB": diff.mean(),
        "RMSE": np.sqrt((diff ** 2).mean()),
        "R": obs.corr(mod),
        "NMB_%": nmb,
        "NME_%": nme,
    })


In [44]:

combined = []

for i in range(len(var)):
    species = var[i]
    code = epa_code[i]
    print("Processing", species)
    df_i = merge_species(species, code)

    # Ensure Datetime GMT is a column (not index) and remove duplicate columns
    if "Datetime GMT" not in df_i.columns:
        df_i = df_i.reset_index().rename(columns={"index": "Datetime GMT"})
    else:
        # just in case Datetime GMT is also in the index from earlier steps
        df_i = df_i.reset_index(drop=True)

    df_i = df_i.loc[:, ~df_i.columns.duplicated()].copy()
    combined.append(df_i)

combined_df = pd.concat(combined, ignore_index=True)

combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()].copy()
combined_df["Datetime GMT"] = pd.to_datetime(combined_df["Datetime GMT"], errors="coerce")

combined_df["in_d03"] = label_in_d03(combined_df)

print("Datetime GMT columns:", (combined_df.columns == "Datetime GMT").sum())
print("Rows in d03:", combined_df["in_d03"].sum())

Processing SO2


C:\Users\x12la\AppData\Local\Temp\ipykernel_16712\831649441.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processing NO2


C:\Users\x12la\AppData\Local\Temp\ipykernel_16712\831649441.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processing O3


...

Processing CO


C:\Users\x12la\AppData\Local\Temp\ipykernel_16712\831649441.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processing PM25_TOT


C:\Users\x12la\AppData\Local\Temp\ipykernel_16712\831649441.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Datetime GMT columns: 1
Rows in d03: 25296


In [45]:
hourly_stats_d03 = (
    combined_df[combined_df["in_d03"]]
    .groupby("Species")
    .apply(calc_metrics)
)

In [46]:
d03_hourly = combined_df[combined_df["in_d03"]].copy()

# IMPORTANT: remove duplicate column names before resample/select
d03_hourly = d03_hourly.loc[:, ~d03_hourly.columns.duplicated()].copy()

# Parse datetime safely
d03_hourly["Datetime GMT"] = pd.to_datetime(d03_hourly["Datetime GMT"], errors="coerce")
d03_hourly = d03_hourly.dropna(subset=["Datetime GMT"]).sort_values("Datetime GMT")

# Sanity check
assert "OBS" in d03_hourly.columns and "CMAQ" in d03_hourly.columns, d03_hourly.columns
assert (d03_hourly.columns == "OBS").sum() == 1
assert (d03_hourly.columns == "CMAQ").sum() == 1

daily_df_d03 = (
    d03_hourly
    .set_index("Datetime GMT")
    .groupby(["Species", "Latitude", "Longitude"])[["OBS", "CMAQ"]]
    .resample("1D")
    .mean()
    .reset_index()
)

daily_stats_d03 = (
    daily_df_d03
    .groupby("Species")
    .apply(calc_metrics)
)


In [47]:
month_tag = start_dt.strftime("%Y%m")

combined_df.to_csv(f"CMAQ_EPA_hourly_allstations_{month_tag}.csv", index=False)

# d03-only outputs
d03_hourly.to_csv(f"CMAQ_EPA_hourly_d03_{month_tag}.csv", index=False)
daily_df_d03.to_csv(f"CMAQ_EPA_daily_d03_{month_tag}.csv", index=False)

hourly_stats_d03.to_csv(f"Hourly_metrics_d03_{month_tag}.csv")
daily_stats_d03.to_csv(f"Daily_metrics_d03_{month_tag}.csv")

print("Done.")
print(hourly_stats_d03)
print(daily_stats_d03)

Done.
                N    Mean_OBS   Mean_CMAQ         MB        RMSE         R  \
Species                                                                      
CO         1483.0  303.774107  220.141501 -83.632605  164.630821  0.471539   
NO2        3211.0   14.961663   11.047539  -3.914124   10.154332  0.591621   
O3        10844.0   36.637588   38.443558   1.805971   12.211918  0.759048   
PM25_TOT   5538.0   13.907945   10.228242  -3.679703   10.665260  0.690006   
SO2        2181.0    1.503576    0.591282  -0.912294    2.093089  0.191894   

              NMB_%      NME_%  
Species                         
CO       -27.531183  37.326850  
NO2      -26.161020  50.024193  
O3         4.929284  25.242437  
PM25_TOT -26.457558  48.123545  
SO2      -60.674956  94.736708  
              N    Mean_OBS   Mean_CMAQ         MB        RMSE         R  \
Species                                                                    
CO         62.0  303.668377  220.062950 -83.605427  123.666804  